In [ ]:

import os
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec
from scipy.interpolate import Rbf, LinearNDInterpolator
import umap

# Add project root to path
sys.path.append(os.path.abspath('../../'))
from BWM.ScriptsandFunctions.bwm_functions import *

projections_cache = {}

# Set global plotting parameters
plt.rcParams["axes.grid"] = False
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 7,
    'axes.titlesize': 7,
    'axes.labelsize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 7,
    'figure.titlesize': 7
})


In [ ]:
# Helper functions
def iso_ae_preprocessing_tensor(response_tensor):
    # Z-score and reshape
    response_tensor_norm = (response_tensor - np.mean(response_tensor, axis=(0, 1), keepdims=True)) / (np.std(response_tensor, axis=(0, 1), keepdims=True) + 1e-8)
    # response_tensor_norm shape: (conditions, time, neurons)
    # Reshape to (conditions * time, neurons) and then transpose to (neurons, features)
    response_matrix = response_tensor_norm.reshape(-1, response_tensor_norm.shape[2])
    X_transposed = response_matrix.T

    # Scale each row (neuron) to unit length
    row_norms = np.linalg.norm(X_transposed, axis=1, keepdims=True)
    X_pre_pca = X_transposed / (row_norms + 1e-8)

    return X_pre_pca

def calculate_msd_data(response_tensor):
    """
    Calculate Mean Standardized Derivative using finite differences.
    responses: (conditions, time, neurons)
    """
    z = (response_tensor - np.mean(response_tensor, axis=(0, 1), keepdims=True)) / (np.std(response_tensor, axis=(0, 1), keepdims=True) + 1e-8)
    speed = np.mean(np.abs(np.diff(z, axis=1)), axis=(0, 1))
    return speed


In [ ]:
# Configuration
directory_path = '../Data2'
event = 'stimOn_times'
conditions = ['choice']
selected_region = 'CP' # Default region


In [ ]:
def iso_ae_preprocessing(pseudo_response_tensor):
    """
    Preprocess neural responses intro response matrix (n x kt) for autoencoder.
    """
    response_tensor = np.mean(pseudo_response_tensor, axis=0)
    # Z-score and reshape
    response_tensor_norm = (response_tensor - np.mean(response_tensor, axis=(0, 1), keepdims=True)) / (np.std(response_tensor, axis=(0, 1), keepdims=True) + 1e-8)
    response_matrix = response_tensor_norm.reshape(-1, response_tensor_norm.shape[2])
    X_transposed = response_matrix.T
    # Scale each row (neuron) to unit length
    #row_norms = np.linalg.norm(X_transposed, axis=1, keepdims=True)
    #X_pre_pca = X_transposed / (row_norms + 1e-8)
    return X_transposed


directory_path='../Data2'
eids = [d for d in os.listdir(directory_path) if os.path.isdir(os.path.join(directory_path, d))]
result = get_region_response_matrix(
                region=['CP'],
                eids=eids,
                directory_path=directory_path,
                event="stimOn_times",
                conditions=['choice'],
                n_pseudo=25,
                threshold=1,
                std_above=50,
                std_below=50,
                filter_significant=False,
                n_steps=100,
                step=0.01,
                sigma=2,
                mapping='Beryl'
            )
pseudo_response_tensor =result[0]

In [ ]:
def trial_variability(x):

    z = (x-np.mean(x,axis=(0,1,2)))/np.std(x,axis=(0,1,2))
    return (
        np.mean(np.std(z, axis=0),axis=(0,1)))

variability = trial_variability(pseudo_response_tensor)
pseudo_response_tensor=pseudo_response_tensor[:,:,:,variability<.7]

X_input = iso_ae_preprocessing(pseudo_response_tensor)

def calculate_msd_data(response_tensor):
    """
    Calculate Mean Standardized Derivative using finite differences.
    responses: (conditions, time, neurons)
    """
    z = (response_tensor - np.mean(response_tensor, axis=(0, 1), keepdims=True)) / (np.std(response_tensor, axis=(0, 1), keepdims=True) + 1e-8)
    speed= np.mean(np.abs(np.diff(z,axis=1)),axis=(0,1))

    return speed

msd_data = calculate_msd_data(np.mean(pseudo_response_tensor, axis=0))


In [ ]:
# Plot densmap projection of EID responses
#Figure 1
fig = plt.figure(figsize=(6, 2))
gs = gridspec.GridSpec(1, 3)
ax1 = plt.subplot(gs[0,0])
ax2 = plt.subplot(gs[0,1])
ax3 = plt.subplot(gs[0,2])
# First row
#from sklearn.manifold import Isomap
z_neurons = umap.UMAP(densmap = True, dens_lambda=2, n_neighbors=500).fit_transform(X_input)
sns.scatterplot(x = z_neurons[:,0], y= z_neurons[:,1], color = 'gray',ax=ax1,s=1,legend=False,alpha =.5)
sns.scatterplot(x = z_neurons[:,0], y= z_neurons[:,1], hue = msd_data,ax=ax2,palette='turbo',s=1,alpha = .5,legend=False)

from scipy.interpolate import Rbf

# Calculate limits for plotting based on the initial state
x_min, x_max = z_neurons[:,0].min(), z_neurons[:,0].max()
y_min, y_max = z_neurons[:,1].min(), z_neurons[:,1].max()
pad = 0.5

# Prepare RBF background
grid_x, grid_y = np.mgrid[x_min-pad:x_max+pad:100j, y_min-pad:y_max+pad:100j]
rbf = Rbf(z_neurons[:,0], z_neurons[:,1], msd_data, function='multiquadric', smooth=1)
grid_msd = rbf(grid_x, grid_y)
sns.heatmap(grid_msd.T,ax=ax3,cmap='Spectral_r')


In [ ]:
# # Region analysis figure 1
#
# # 1. Aggregate CP neurons across all EIDs
# #all_eids = [d for d in os.listdir(directory_path) if os.path.isdir(os.path.join(directory_path, d))]
# #print(f"Found {len(all_eids)} EIDs in total.")
#
# # pr_tensor, cond_matrix, _ = get_region_response_matrix(
# #     selected_region, all_eids, directory_path, event, conditions,
# #     n_pseudo=25, threshold=5, std_above=50, std_below=50,
# #     filter_significant=False, n_steps=100, step=0.01, sigma=2
# # )
#
#
# pr_tensor = np.load('/Users/cl1704/PycharmProjects/embedding_dimension/BWM/ResponseData/CP_pseudo_responses.npy')
# pr_tensor = pr_tensor[:,:,:,np.mean(pr_tensor,axis=(0,1,2))<15]
#
#
# if pr_tensor is not None:
#     print(f"Successfully aggregated {pr_tensor.shape[3]} neurons in {selected_region} across EIDs.")
#
#     # 2. Preprocessing & densMAP
#     # Average across pseudo-trials
#     avg_resp = np.mean(pr_tensor, axis=0) # (conditions, time, neurons)
#     X_iso = iso_ae_preprocessing_tensor(avg_resp)
#
#     reducer = umap.UMAP(densmap=True, dens_lambda=2, n_neighbors=min(500, X_iso.shape[0]-1), n_jobs=1, random_state=42)
#     proj = reducer.fit_transform(X_iso)
#
#     # Cache for consistency
#     projections_cache['aggregated'] = proj
#
#     # 3. MSD computation
#     msd_vals = calculate_msd_data(avg_resp)
#
#     # 3.5 RBF Interpolation setup
#     x_min, x_max = proj[:, 0].min(), proj[:, 0].max()
#     y_min, y_max = proj[:, 1].min(), proj[:, 1].max()
#     xi = np.linspace(x_min, x_max, 200)
#     yi = np.linspace(y_min, y_max, 200)
#     XI, YI = np.meshgrid(xi, yi)
#
#     try:
#         rbf_obj = Rbf(proj[:, 0], proj[:, 1], msd_vals, function='multiquadric', smooth=10)
#         ZI = rbf_obj(XI, YI)
#         vmin, vmax = np.nanmin(msd_vals), np.nanmax(msd_vals)
#         #vmin = max(vmin, 0.05)
#     except Exception as e:
#         print(f"RBF Pre-calculation failed: {e}")
#         ZI = None
#
#     # 4. Create Figure: densMAP Projections, RBF Overlay, and 3D Surfaces
#     fig = plt.figure(figsize=(16, 3.5))
#     gs = gridspec.GridSpec(1, 4, wspace=0.3)
#
#     # --- 1. 2D densMAP Projection (Colored by MSD) ---
#     ax_1 = fig.add_subplot(gs[0, 0])
#     sns.scatterplot(x=proj[:, 0], y=proj[:, 1], color='gray', s=1, alpha=0.5, ax=ax_1, legend=False)
#     ax_1.set_title("densMAP Projection (MSD)")
#     ax_1.set_xticks([])
#     ax_1.set_yticks([])
#     sns.despine(ax=ax_1, left=True, bottom=True)
#
#     # --- 2. 3D MSD Surface Bird's Eye View ---
#     ax_2 = fig.add_subplot(gs[0, 1], projection='3d')
#     if ZI is not None:
#         try:
#             ax_2.plot_surface(XI, YI, ZI, cmap='Spectral_r', edgecolor='none',
#                               alpha=0.9, vmin=vmin, vmax=vmax, antialiased=True,
#                               rcount=200, ccount=200)
#             ax_2.set_title("3D MSD Surface (Bird's Eye)")
#             ax_2.view_init(elev=90, azim=-90)
#             ax_2.set_axis_off()
#             ax_2.set_box_aspect(None, zoom=1.2)
#         except Exception as e:
#             print(f"Bird's Eye 3D Plot failed: {e}")
#
#     # --- 3. 3D MSD Surface Plots ---
#     ax_3 = fig.add_subplot(gs[0, 2], projection='3d')
#     try:
#         # First 3D plot
#         surf = ax_3.plot_surface(XI, YI, ZI, cmap='Spectral_r', edgecolor='none',
#                                  alpha=0.9, vmin=vmin, vmax=vmax, antialiased=True,
#                                  rcount=200, ccount=200)
#         ax_3.set_title("3D MSD Surface (Angle 1)")
#         ax_3.view_init(elev=30, azim=-60)
#         ax_3.set_axis_off()
#         ax_3.set_box_aspect(None, zoom=1.2)
#
#         # Second 3D plot with different angle
#         ax_4 = fig.add_subplot(gs[0, 3], projection='3d')
#         ax_4.plot_surface(XI, YI, ZI, cmap='Spectral_r', edgecolor='none',
#                           alpha=0.9, vmin=vmin, vmax=vmax, antialiased=True,
#                           rcount=200, ccount=200)
#         ax_4.set_title("3D MSD Surface (Angle 2)")
#         ax_4.view_init(elev=80, azim=60)
#         ax_4.set_axis_off()
#         ax_4.set_box_aspect(None, zoom=1.2)
#     except Exception as e:
#         print(f"3D Plot failed: {e}")
#
#     #plt.tight_layout()
#     os.makedirs('../Figures', exist_ok=True)
#     plt.savefig(f'../Figures/msd_rbf_overlay_{selected_region}.png', dpi=300, bbox_inches='tight')
#     plt.show()
# else:
#     print(f"Failed to aggregate data for region {selected_region}")


In [ ]:
pr_tensor = pseudo_response_tensor

In [ ]:
# 2. Spatiotemporal visualization of activity on densMAP projection (Condition 1 & 2) - Filtered by Region (Aggregated)
print(f"Visualizing activity over time for Aggregated Region: {selected_region}")

if pr_tensor is not None:
    # Use aggregated data already loaded
    # Average across pseudo-trials
    avg_resp = np.mean(pr_tensor, axis=0) # (conditions, time, neurons)
    
    # Preprocessing & densMAP
    if 'aggregated' in projections_cache:
        print(f"Using cached projection for aggregated {selected_region}")
        proj = projections_cache['aggregated']
    else:
        X_iso = iso_ae_preprocessing_tensor(avg_resp)
        reducer = umap.UMAP(densmap=True, dens_lambda=2, n_neighbors=min(50, X_iso.shape[0]-1), n_jobs=1, random_state=42)
        proj = reducer.fit_transform(X_iso)
        projections_cache['aggregated'] = proj
    
    # Loop over conditions (Condition 1 and 2)
    for cond_idx in range(min(2, avg_resp.shape[0])):
        print(f"Visualizing Condition {cond_idx + 1}")
        activity = avg_resp[cond_idx, :, :] # (time, neurons)
        
        # Z-score each neuron's activity across time
        activity_z = (activity - np.mean(activity, axis=0, keepdims=True)) / (np.std(activity, axis=0, keepdims=True) + 1e-8)
        
        # Select 5 time points
        n_time = activity.shape[0]
        t_indices = np.linspace(0, n_time - 1, 5, dtype=int)
        
        fig, axes = plt.subplots(1, 5, figsize=(15, 4), sharex=True, sharey=True)
        axes = axes.flatten()
        
        # Fixed color scale across all plots in each condition
        vabs = np.max(np.abs(activity_z))
        vmin, vmax = -2, 2
        
        cmap ='cividis'

        sc = None
        for i, t_idx in enumerate(t_indices):
            ax = axes[i]
            c = activity_z[t_idx, :]
            
            # Use sns.scatterplot but keep the scatter object for colorbar if needed
            sns.scatterplot(x=proj[:, 0], y=proj[:, 1], hue=c, palette=cmap, s=5, alpha=0.8, hue_norm=(vmin, vmax), ax=ax, legend=False)
            sc = ax.collections[-1]

            #ax.scatter(proj[:, 0], proj[:, 1], c='k', s=1, alpha=0.1) # Overlay neurons
            ax.set_title(f"Time Point {t_idx}\n(t = {t_idx * 0.005:.3f}s)")
            ax.set_xticks([])
            ax.set_yticks([])
            sns.despine(ax=ax, left=True, bottom=True)
            
            if i == 4:
                if sc is not None:
                    plt.colorbar(sc, ax=ax, label='Z-scored Activity')
        
        plt.suptitle(f"Neural Activity Over Time (Condition {cond_idx + 1}) on densMAP Projection (Aggregated Region: {selected_region})", fontsize=14)
        plt.tight_layout()
        os.makedirs('../Figures', exist_ok=True)
        plt.savefig(f'../Figures/activity_over_time_{selected_region}_cond{cond_idx+1}.png', dpi=300, bbox_inches='tight')
        plt.show()

else:
    print(f"Aggregated data not available for visualization.")


In [ ]:
# 3. Side-by-side Animation of Spatiotemporal Dynamics for both conditions
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

if pr_tensor is not None:
    print("Generating side-by-side animation...")
    
    # Average across pseudo-trials if not already done
    avg_resp = np.mean(pr_tensor, axis=0) # (conditions, time, neurons)
    n_time = avg_resp.shape[1]
    
    # Use cached projection
    if 'aggregated' in projections_cache:
        proj = projections_cache['aggregated']
    else:
        X_iso = iso_ae_preprocessing_tensor(avg_resp)
        reducer = umap.UMAP(densmap=True, dens_lambda=2, n_neighbors=min(50, X_iso.shape[0]-1), n_jobs=1, random_state=42)
        proj = reducer.fit_transform(X_iso)
        projections_cache['aggregated'] = proj

    # Z-score each neuron's activity across time for both conditions to get consistent scale
    activity_z_all = []
    for c_idx in range(min(2, avg_resp.shape[0])):
        act = avg_resp[c_idx, :, :]
        act_z = (act - np.mean(act, axis=0, keepdims=True)) / (np.std(act, axis=0, keepdims=True) + 1e-8)
        activity_z_all.append(act_z)
    
    # Calculate global vmin/vmax for animation across both conditions
    vabs_max = max([np.max(np.abs(az)) for az in activity_z_all])
    vmin, vmax = -2, 2 # Consistent with static plots
    cmap = 'cividis'
    
    fig_anim, axes_anim = plt.subplots(1, 2, figsize=(15, 7), sharex=True, sharey=True)
    
    def update(t_idx):
        results = []
        for c_idx in range(len(activity_z_all)):
            ax = axes_anim[c_idx]
            ax.clear()
            c = activity_z_all[c_idx][t_idx, :]
            
            sns.scatterplot(x=proj[:, 0], y=proj[:, 1], hue=c, palette=cmap, s=5, alpha=0.8, hue_norm=(vmin, vmax), ax=ax, legend=False)
            sc = ax.collections[-1]
            results.append(sc)
            
            # ax.scatter(proj[:, 0], proj[:, 1], c='k', s=1, alpha=0.1) # Optional: Overlay neurons
            ax.set_title(f"Condition {c_idx + 1}\nTime {t_idx} (t = {t_idx * 0.005:.3f}s)")
            ax.set_xticks([])
            ax.set_yticks([])
            sns.despine(ax=ax, left=True, bottom=True)
        
        fig_anim.suptitle(f"Spatiotemporal Dynamics (Aggregated {selected_region})", fontsize=14)
        plt.tight_layout()
        return results

    anim = FuncAnimation(fig_anim, update, frames=n_time, interval=100)
    
    # Save as GIF
    os.makedirs('../Movies', exist_ok=True)
    save_path = f'../Movies/spatiotemporal_dynamics_aggregated_{selected_region}.gif'
    anim.save(save_path, writer=PillowWriter(fps=10))
    
    # Display in notebook
    display(HTML(anim.to_jshtml()))
    plt.close(fig_anim)
    print(f"Animation saved to {save_path}")
else:
    print(f"Aggregated data not available for animation.")


In [ ]:
# 4. Rotated Neural Space Animation
from scipy.stats import ortho_group
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

if pr_tensor is not None:
    print("\nPerforming Random Rotation and Generating Animation...")
    
    # 1. Get original neural space (neurons, features)
    avg_resp = np.mean(pr_tensor, axis=0) # (conditions, time, neurons)
    X_orig = iso_ae_preprocessing_tensor(avg_resp)
    n_neurons, n_features = X_orig.shape
    
    # 2. Perform a single random rotation using Cayley transform
    # Generate a random matrix
    M = np.random.randn(n_neurons, n_neurons)
    # Create a skew-symmetric matrix A
    A = M - M.T
    # R = (I - A)(I + A)^-1
    I = np.eye(n_neurons)
    R = (I - A) @ np.linalg.inv(I + A)
    
    # Rotate neural space
    X_rot = R @ X_orig
    
    # 3. Re-z-score after rotation
    X_rot_norm = (X_rot - np.mean(X_rot, axis=1, keepdims=True)) / (np.std(X_rot, axis=1, keepdims=True) + 1e-8)
    
    # 4. Apply densMAP
    reducer_rot = umap.UMAP(densmap=True, dens_lambda=2, n_neighbors=min(500, n_neurons-1), n_jobs=1, random_state=42)
    proj_rot = reducer_rot.fit_transform(X_rot_norm)
    
    # 5. Prepare activity for animation (using first condition)
    # The activity should also be rotated? 
    # Usually "activity for each moment in time" refers to the features used for projection.
    # X_rot_norm has shape (n_neurons, n_conditions * n_time)
    n_conditions = avg_resp.shape[0]
    n_time = avg_resp.shape[1]
    
    # Reshape back to (n_neurons, n_conditions, n_time)
    X_rot_reshaped = X_rot_norm.reshape(n_neurons, n_conditions, n_time)
    
    # We'll animate Condition 1
    cond_idx = 0
    activity_to_plot = X_rot_reshaped[:, cond_idx, :].T # (n_time, n_neurons)
    
    fig_rot, ax_rot = plt.subplots(figsize=(8, 8))
    
    # Determine color limits
    vmin, vmax = -2, 2
    cmap = 'cividis'
    
    # Initial plot
    sc = ax_rot.scatter(proj_rot[:, 0], proj_rot[:, 1], c=activity_to_plot[0, :], 
                        cmap=cmap, s=5, alpha=0.8, vmin=vmin, vmax=vmax)
    
    cbar = plt.colorbar(sc, ax=ax_rot, label='Z-scored Activity (after rotation)')
    title = ax_rot.set_title(f"Rotated Neural Space densMAP - Condition {cond_idx+1}, Time 0")
    ax_rot.set_xticks([])
    ax_rot.set_yticks([])
    sns.despine(ax=ax_rot, left=True, bottom=True)

    def update_rot(t_idx):
        sc.set_array(activity_to_plot[t_idx, :])
        title.set_text(f"Rotated Neural Space densMAP - Condition {cond_idx+1}, Time {t_idx} (t = {t_idx * 0.005:.3f}s)")
        return [sc, title]

    anim_rot = FuncAnimation(fig_rot, update_rot, frames=n_time, interval=100, blit=True)
    
    # Save animation
    os.makedirs('../Movies', exist_ok=True)
    save_path = f'../Movies/rotated_densmap_activity_{selected_region}.mp4'
    # Use ffmpeg for mp4
    try:
        anim_rot.save(save_path, writer='ffmpeg', fps=10)
        print(f"Animation saved to {save_path}")
    except Exception as e:
        print(f"Failed to save as mp4, trying gif: {e}")
        save_path = f'../Movies/rotated_densmap_activity_{selected_region}.gif'
        anim_rot.save(save_path, writer=PillowWriter(fps=10))
        print(f"Animation saved to {save_path}")
    
    # Display in notebook
    display(HTML(anim_rot.to_jshtml()))
    plt.close(fig_rot)

else:
    print("Aggregated data not available for rotation animation.")
